# Generate Precomputed Magnetic Field (GPU Analytical)

Generates a precomputed magnetic field lookup table using GPU-accelerated analytical formulas (Engel-Herbert & Hesjedal).

## Magnetization Direction
The magnet is a cube centered at the origin. By default, **magnetization points along +Y axis**:
- North pole face on +Y side
- South pole face on -Y side

**To change direction**, modify `polarization` in the code:
- `[M, 0, 0]` = North pole on +X face
- `[0, M, 0]` = North pole on +Y face (default)
- `[0, 0, M]` = North pole on +Z face


In [1]:
import numpy as np
import cupy as cp
import time
from datetime import datetime
import os
import sys

# Import analytical GPU module
sys.path.insert(0, 'Magnet_design')
from analytical_magnet_gpu import cuboid_field_gpu

print("="*80)
print("GPU ANALYTICAL PRECOMPUTED FIELD GENERATOR")
print("="*80)
print(f"CuPy version: {cp.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"GPU: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")

GPU ANALYTICAL PRECOMPUTED FIELD GENERATOR
CuPy version: 13.4.1
NumPy version: 1.26.4
Date: 2026-10-06 17:13:41
GPU: NVIDIA GeForce RTX 4070


## Configure Parameters (Example Values - Modify as Needed)

In [2]:
# ============================================================
# EXAMPLE VALUES - Modify these for your specific application
# ============================================================

# Material properties (example: N52 NdFeB)
Br = 1.48  # T - Remanence (adjust for your magnet grade)
demag_factor = 0.9836  # Demagnetization correction (geometry dependent)
effective_magnetization = Br * demag_factor  # T

# Magnet geometry (example: 6.35mm cube)
magnet_size_mm = 6.35  # mm - Change to your magnet size
magnet_size_m = magnet_size_mm * 1e-3  # m

# Grid parameters (example: 450mm span at 1mm resolution)
grid_range_mm = 100  # mm (±100 mm = 200mm   total) - Adjust for your FOV
step_size_mm = 1.0   # mm - Smaller = more accurate but larger file

# Batch size for GPU processing
batch_size = 500000  # points per batch - reduce if GPU runs out of memory
# Recommended batch sizes:
#   1,000,000 = fastest, needs ~8 GB VRAM
#   500,000 = balanced, needs ~4 GB VRAM (DEFAULT)
#   100,000 = safe for smaller GPUs

# ============================================================
# Convert to meters (don't modify)
# ============================================================
grid_range_m = grid_range_mm * 1e-3
step_size_m = step_size_mm * 1e-3

print("\n" + "="*80)
print("CONFIGURATION (Example Values - Modify Above)")
print("="*80)
print(f"\nMaterial Properties:")
print(f"  • Remanence (Br): {Br} T")
print(f"  • Demagnetization Factor: {demag_factor}")
print(f"  • Effective Magnetization: {effective_magnetization:.6f} T")
print(f"\nMagnet Geometry:")
print(f"  • Size: {magnet_size_mm} mm cube")
print(f"  • Polarization: +Y axis (0, {effective_magnetization:.6f}, 0)")
print(f"\nGrid Parameters:")
print(f"  • Range: ±{grid_range_mm} mm ({grid_range_mm*2} mm total)")
print(f"  • Step Size: {step_size_mm} mm")

# Calculate grid size
n_points_per_axis = int(2 * grid_range_m / step_size_m)
total_points = n_points_per_axis ** 3
n_batches = (total_points + batch_size - 1) // batch_size

print(f"  • Points per axis: {n_points_per_axis}")
print(f"  • Total grid points: {total_points:,}")
print(f"\nMemory Estimates:")
print(f"  • Final file size (compressed): ~{total_points * 3 * 8 / (1024**3) * 0.6:.2f} GB")
print(f"  • Number of batches: {n_batches:,}")
print(f"  • Batch size: {batch_size:,} points")
print("="*80)


CONFIGURATION (Example Values - Modify Above)

Material Properties:
  • Remanence (Br): 1.48 T
  • Demagnetization Factor: 0.9836
  • Effective Magnetization: 1.455728 T

Magnet Geometry:
  • Size: 6.35 mm cube
  • Polarization: +Y axis (0, 1.455728, 0)

Grid Parameters:
  • Range: ±100 mm (200 mm total)
  • Step Size: 1.0 mm
  • Points per axis: 200
  • Total grid points: 8,000,000

Memory Estimates:
  • Final file size (compressed): ~0.11 GB
  • Number of batches: 16
  • Batch size: 500,000 points


## Generate Grid

In [3]:
print("\nGenerating grid...")
start_time = time.time()

# Create 1D grid
grid_range = np.arange(-grid_range_m, grid_range_m, step_size_m)
print(f"  • 1D grid: {len(grid_range)} points")

# Create 3D meshgrid
X, Y, Z = np.meshgrid(grid_range, grid_range, grid_range, indexing='ij')
Ref_grid = X  # Save for output

# Flatten to points array
points = np.stack((X, Y, Z), axis=-1).reshape(-1, 3)

grid_time = time.time() - start_time
print(f"✓ Grid generated in {grid_time:.2f} seconds")
print(f"  • Shape: {X.shape}")
print(f"  • Total points: {len(points):,}")
print(f"  • Points array size: {points.nbytes / (1024**2):.1f} MB")

# Clean up large arrays we don't need anymore
del Y, Z
print(f"  • Cleaned up temporary arrays")


Generating grid...
  • 1D grid: 200 points
✓ Grid generated in 0.13 seconds
  • Shape: (200, 200, 200)
  • Total points: 8,000,000
  • Points array size: 183.1 MB
  • Cleaned up temporary arrays


## Create Magnet (Using Parameters Above)

In [4]:
print("\nConfiguring magnet parameters...")

# ============================================================
# MAGNETIZATION DIRECTION - Modify to change North pole face
# ============================================================
# Default: +Y axis (North pole on +Y face)
# Options:
#   [M, 0, 0] = North pole on +X face
#   [0, M, 0] = North pole on +Y face (default)
#   [0, 0, M] = North pole on +Z face
# ============================================================

polarization = [0, effective_magnetization, 0]  # Change this to set direction
dimensions = [magnet_size_m, magnet_size_m, magnet_size_m]

# Determine which axis from polarization
if polarization[0] != 0:
    axis = "+X"
elif polarization[1] != 0:
    axis = "+Y"
else:
    axis = "+Z"

print(f"  • Magnetization direction: {axis} axis (North pole face on {axis} side)")
print(f"  • Polarization vector: ({polarization[0]:.4f}, {polarization[1]:.4f}, {polarization[2]:.4f}) T")
print(f"  • Dimensions: ({magnet_size_mm}, {magnet_size_mm}, {magnet_size_mm}) mm cube")
print(f"  • Position: Origin (0, 0, 0)")
print("✓ Parameters configured for GPU analytical calculation")


Configuring magnet parameters...
  • Magnetization direction: +Y axis (North pole face on +Y side)
  • Polarization vector: (0.0000, 1.4557, 0.0000) T
  • Dimensions: (6.35, 6.35, 6.35) mm cube
  • Position: Origin (0, 0, 0)
✓ Parameters configured for GPU analytical calculation


## Calculate Field in Batches

In [5]:
print("\n" + "="*80)
print("CALCULATING MAGNETIC FIELD (GPU ANALYTICAL)")
print("="*80)
print(f"Processing {total_points:,} points in {n_batches:,} batches...\n")

# Pre-allocate output array
Bxyz = np.zeros((total_points, 3), dtype=np.float64)

calc_start = time.time()

for i in range(0, total_points, batch_size):
    end = min(i + batch_size, total_points)
    batch_num = i // batch_size + 1
    
    # Calculate field for this batch using GPU analytical
    batch_points = cp.asarray(points[i:end], dtype=cp.float64)
    B_gpu = cuboid_field_gpu(batch_points, polarization, dimensions)
    Bxyz[i:end] = cp.asnumpy(B_gpu)
    
    # Free GPU memory
    del batch_points, B_gpu
    cp.get_default_memory_pool().free_all_blocks()
    
    # Progress update every 10 batches or at end
    current_time = time.time()
    if batch_num % 10 == 0 or end == total_points:
        elapsed = current_time - calc_start
        progress_pct = end / total_points * 100
        rate = end / elapsed if elapsed > 0 else 0
        eta = (total_points - end) / rate if rate > 0 else 0
        
        print(f"  Batch {batch_num:>4}/{n_batches}: "
              f"{end:>10,}/{total_points:,} points "
              f"({progress_pct:>5.1f}%) | "
              f"Rate: {rate:>8,.0f} pts/s | "
              f"ETA: {eta/60:>5.1f} min")

calc_time = time.time() - calc_start

print(f"\n✓ Field calculation complete!")
print(f"  • Total time: {calc_time:.1f} seconds ({calc_time/60:.1f} minutes)")
print(f"  • Average rate: {total_points/calc_time:,.0f} points/second")
print("="*80)


CALCULATING MAGNETIC FIELD (GPU ANALYTICAL)
Processing 8,000,000 points in 16 batches...



  Batch   10/16:  5,000,000/8,000,000 points ( 62.5%) | Rate: 10,326,266 pts/s | ETA:   0.0 min
  Batch   16/16:  8,000,000/8,000,000 points (100.0%) | Rate: 11,555,189 pts/s | ETA:   0.0 min

✓ Field calculation complete!
  • Total time: 0.7 seconds (0.0 minutes)
  • Average rate: 11,553,017 points/second


## Save Precomputed Field

In [6]:
# Save precomputed field
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"Precalc_field_{int(grid_range_mm*2)}mm_grid_{int(step_size_mm)}mm_step_Br{Br}_demag{demag_factor}_{timestamp}.npz"

np.savez_compressed(filename, Bxyzvalue=Bxyz, Ref_grid=Ref_grid, Grid_dim=grid_range, step_size=step_size_m)

file_size_gb = os.path.getsize(filename) / (1024**3)
print(f"✓ Saved: {filename} ({file_size_gb:.2f} GB)")

# Cleanup
del Bxyz, points, Ref_grid

✓ Saved: Precalc_field_200mm_grid_1mm_step_Br1.48_demag0.9836_20261006_171342.npz (0.13 GB)
